In [1]:
# %% Shared setup
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "Tapiculin"
STUDENT_ID = "TUPM-26-1932"
id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)

WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

X = np.genfromtxt(DATA_DIR / "secom.data", dtype=float)          # measurement matrix
labels = np.loadtxt(DATA_DIR / "secom_labels.data", usecols=0, dtype=int)
assert len(labels) == len(X)
feature_names = [f"Feature_{i:03d}" for i in range(X.shape[1])]

missing = np.isnan(X)
observed_n = (~missing).sum(axis=0)
obs_min = np.where(missing, np.inf, X).min(axis=0)
obs_max = np.where(missing, -np.inf, X).max(axis=0)
all_missing = observed_n == 0
constant = (observed_n > 0) & (obs_min == obs_max)
usable = np.flatnonzero(~(all_missing | constant))               # at least two distinct observed values
print("Records:", X.shape[0], "| Features:", X.shape[1], "| Usable features:", len(usable))
print("Seed digit:", SEED_DIGIT, "| ID sum:", ID_SUM, "| Random seed:", RANDOM_SEED)


# %% EXERCISE 3 - Statistical Robustness and Sensitivity Analysis
rng = np.random.default_rng(RANDOM_SEED)
is_fail = labels == 1
obs_pass = (~missing[~is_fail]).sum(axis=0)
obs_fail = (~missing[is_fail]).sum(axis=0)
eligible = np.flatnonzero(~(all_missing | constant) & (obs_pass >= 10) & (obs_fail >= 10))
feat_pos = np.sort(rng.choice(eligible, size=5, replace=False))
feats = [feature_names[i] for i in feat_pos]
BOOT = 200 + 10 * SEED_DIGIT

def prepare(strategy):
    sub = X[:, feat_pos].copy()
    if strategy == "median_imputation":
        sub = np.where(np.isnan(sub), np.nanmedian(sub, axis=0), sub)      # medians from the full dataset
    return sub                                                              # available_case keeps NaN

def group_values(sub, j, group):
    v = sub[is_fail == group, j]
    return v[~np.isnan(v)]

def boot_ci(sub, j, seed):
    r = np.random.default_rng(seed)
    f, p = group_values(sub, j, True), group_values(sub, j, False)
    diffs = np.empty(BOOT)
    for b in range(BOOT):                                    # independent resampling within each outcome group
        diffs[b] = r.choice(f, f.size, replace=True).mean() - r.choice(p, p.size, replace=True).mean()
    return diffs.mean(), np.percentile(diffs, 2.5), np.percentile(diffs, 97.5)

def run(seed):
    out = []
    for strat in ("available_case", "median_imputation"):
        sub = prepare(strat)
        for j, name in enumerate(feats):
            obs_diff = group_values(sub, j, True).mean() - group_values(sub, j, False).mean()
            bm, lo, hi = boot_ci(sub, j, seed + 1000 * j + (7 if strat == "median_imputation" else 0))
            out.append({"Feature": name, "Strategy": strat, "ObservedDiff": obs_diff,
                        "BootMeanDiff": bm, "CI95_Low": lo, "CI95_High": hi, "CI_Excludes_0": bool(lo > 0 or hi < 0)})
    return pd.DataFrame(out)

result = run(RANDOM_SEED)
reproducible = result.equals(run(RANDOM_SEED))                      # same seed -> identical output

wide = result.pivot(index="Feature", columns="Strategy", values="ObservedDiff")
sign_changed = [f for f in feats if np.sign(wide.loc[f, "available_case"]) != np.sign(wide.loc[f, "median_imputation"])]
rank_ac = wide["available_case"].abs().rank(ascending=False)
rank_mi = wide["median_imputation"].abs().rank(ascending=False)
rank_changed = [f for f in feats if rank_ac[f] != rank_mi[f]]
shift = (wide["available_case"] - wide["median_imputation"]).abs()

result.round(4).to_csv(OUTPUT_DIR / "robustness_comparison.csv", index=False)
report = (
    "STATISTICAL ROBUSTNESS AND SENSITIVITY REPORT\n"
    f"Student: {LAST_NAME} {STUDENT_ID} | Random seed: {RANDOM_SEED} | Bootstrap repetitions: {BOOT}\n"
    f"Selected features: {', '.join(feats)}\n"
    f"Features with changed difference sign: {sign_changed or 'none'}\n"
    f"Features with changed ranking: {rank_changed or 'none'}\n"
    f"Largest change in mean difference: {shift.idxmax()} ({shift.max():.4f})\n"
    f"Reproducibility verified: {reproducible}\n\n"
    "Interpretation: differences are descriptive; overlapping confidence intervals or a sign change\n"
    "between strategies indicate a conclusion that is sensitive to preprocessing choices.\n"
)
(OUTPUT_DIR / "sensitivity_report.txt").write_text(report, encoding="utf-8")

print(result.round(4).to_string(index=False))
print("\n=== Assessment Data ===")
print("Selected Features:", feats)
print("Bootstrap Repetitions:", BOOT)
print("Largest Change in Mean Difference:", shift.idxmax(), round(float(shift.max()), 4))
print("Features with Changed Difference Sign:", sign_changed or "none")
print("Features with Changed Ranking:", rank_changed or "none")
print("Reproducibility Verified:", reproducible)
print("Required Outputs: robustness_comparison.csv, sensitivity_report.txt")


Records: 1567 | Features: 590 | Usable features: 474
Seed digit: 2 | ID sum: 23 | Random seed: 261932
    Feature          Strategy  ObservedDiff  BootMeanDiff  CI95_Low  CI95_High  CI_Excludes_0
Feature_000    available_case       -7.4211       -8.3728  -24.9244    10.5841          False
Feature_055    available_case       -0.9326       -0.8749   -6.1459     5.1732          False
Feature_107    available_case       -0.0053       -0.0062   -0.0264     0.0127          False
Feature_272    available_case        2.2201        2.1066   -1.0675     5.5387          False
Feature_437    available_case        0.4548        0.4625   -0.1016     1.2928          False
Feature_000 median_imputation       -7.4069       -7.4991  -23.0937     9.5893          False
Feature_055 median_imputation       -0.9405       -1.2264   -7.3675     4.8573          False
Feature_107 median_imputation       -0.0054       -0.0059   -0.0253     0.0124          False
Feature_272 median_imputation        2.1925        2